In [0]:
EH_NAMESPACE = "evhns-clickstream-stu" 
EH_TOPIC = "clickstream-topic"
BOOTSTRAP_SERVERS = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
STORAGE_ACCOUNT_NAME = "stclickstreamstu"
CONTAINER_NAME = "datalake"
EH_CONNECTION_STRING = dbutils.secrets.get(scope="clickstream_secrets", key="eventhub-conn-string")
EH_SASL = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONNECTION_STRING}";'

# Đường dẫn Data Lake (Unity Catalog sẽ tự động cấp quyền truy cập vào đây)
BRONZE_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/bronze/events"
CHECKPOINT_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/bronze_events"

print("Đã load xong cấu hình!")

In [0]:
from pyspark.sql.functions import current_timestamp, col

print("Đang kết nối luồng dữ liệu từ Event Hubs...")

raw_stream_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", EH_TOPIC)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", EH_SASL)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false") # Bỏ qua lỗi nếu Event Hubs xóa data cũ
    .load()
)
# Bung dữ liệu nhị phân thành chuỗi string và tạo thời gian
bronze_df = raw_stream_df.select(
    col("value").cast("string").alias("raw_payload"),
    current_timestamp().alias("ingestion_timestamp")
)
print("Cấu hình luồng đọc thành công!")

In [0]:
print(f"Đang hút dữ liệu và ghi xuống Data Lake tại:\n{BRONZE_TABLE_PATH}")

query_bronze = (bronze_df.writeStream
    .format("delta") 
    .outputMode("append") 
    .option("checkpointLocation", CHECKPOINT_PATH) 
    .trigger(availableNow=True) # Thay thế trigger bằng AvailableNow để phù hợp với cluster
    .start(BRONZE_TABLE_PATH)
)
query_bronze.awaitTermination()
print("Đã hoàn tất batch dữ liệu!")

In [0]:
try:
    df_test = spark.read.format("delta").load(BRONZE_TABLE_PATH)
    total_rows = df_test.count()
    print(f"Tổng số dòng hiện có trong Bronze: {total_rows}")
    display(df_test.orderBy(col("ingestion_timestamp").desc()).limit(10))
except Exception as e:
    print(f"Có lỗi xảy ra khi đọc file: {e}")